# Inferência com limiar de confiança

Classifica imagens novas com um modelo de `modelos_pt/`, usando o mesmo pré-processamento do treino. Se a maior probabilidade ficar abaixo do limiar, o resultado é **inconclusivo: encaminhar ao médico**.

## Como usar

1. Na célula de configuração, escolha `arquivo_modelo` e `limiar`.
2. Coloque as imagens em `data/imagens_novas/` e rode o notebook. Não precisa de GPU.
3. Confira o teste de consistência: ele precisa passar antes de confiar nas previsões.
4. Use a curva de acurácia x cobertura para escolher o limiar. As imagens de `data/teste_externo/<cancer>/<classe>/` entram na curva, se existirem.

In [ ]:
import re
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import Tensor
from torch.nn import AdaptiveAvgPool2d, Conv2d, Dropout, Flatten, Linear, MaxPool2d, Module, ReLU, Sequential

In [ ]:
arquivo_modelo = "mama_cor.pt"
limiar = 0.8

# Docker
# modelos_dir = Path("/app/modelos_pt")
# data_dir = Path("/app/data")
# VS Code (PC)
modelos_dir = Path("../modelos_pt")
data_dir = Path("../data")

pasta_imagens = data_dir / "imagens_novas"
teste_externo_dir = data_dir / "teste_externo"

# Imagens por classe usadas na curva, a partir do fim do limite do treino
reserva_por_classe = 100

MENSAGEM_INCONCLUSIVO = "inconclusivo – encaminhar ao médico"
EXTENSOES = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

# Câncer do modelo -> (pasta em data/, prefixo das subpastas de classe)
PASTAS_DATASET = {
    "Cérebro": ("Brain_Cancer", "brain"),
    "Colo do útero": ("Cervical_Cancer", "cervix"),
    "Cólon": ("Colon_Cancer", "colon"),
    "Linfoma": ("Lymphoma", "lymph"),
    "Mama": ("Breast_Cancer", "breast"),
    "Oral": ("Oral_Cancer", "oral"),
    "Pulmão": ("Lung_Cancer", "lung"),
    "Rim": ("Kidney_Cancer", "kidney"),
}

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class CustomModel(Module):
    # Mesma arquitetura dos notebooks; canais e classes vêm do modelo salvo
    def __init__(self, canais: int, output_size: int) -> None:
        super().__init__()

        self.net = Sequential(
            Conv2d(canais, 16, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
            Conv2d(16, 32, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
            Conv2d(32, 64, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
            AdaptiveAvgPool2d(1),
            Flatten(),
            Dropout(p=0.25),
            Linear(64, output_size),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


def carregar_modelo(caminho):
    caminho = Path(caminho)
    if not caminho.exists():
        raise FileNotFoundError(f"Modelo não encontrado: {caminho.resolve()}")
    dados = torch.load(caminho, map_location="cpu", weights_only=True)
    faltando = [chave for chave in ("state_dict", "class_names", "size", "modo", "escala") if chave not in dados]
    if faltando:
        raise ValueError(f"{caminho.name}: faltam os metadados {faltando}")

    state_dict = dados["state_dict"]
    canais = state_dict["net.0.weight"].shape[1]
    n_classes = state_dict["net.12.weight"].shape[0]
    canais_esperados = 3 if dados["modo"] == "cor" else 1
    if canais != canais_esperados:
        raise ValueError(f"{caminho.name}: modo {dados['modo']} pede {canais_esperados} canal(is), mas o state_dict tem {canais}")
    if n_classes != len(dados["class_names"]):
        raise ValueError(f"{caminho.name}: {len(dados['class_names'])} nomes de classe, mas o state_dict tem {n_classes} saídas")

    meta = {
        "class_names": list(dados["class_names"]),
        "size": dados["size"],
        "modo": dados["modo"],
        "escala": dados["escala"],
        "cancer": dados.get("cancer"),
        "id_cancer": re.sub(r"_(cor|cinza).*$", "", caminho.stem),
        "metricas": dados.get("metricas", {}),
        # Modelos antigos não guardam estes campos. Todos os notebooks de cor usam BGR -> RGB; os de cinza não convertem
        "conversao_cor": dados.get("conversao_cor", "BGR2RGB" if dados["modo"] == "cor" else None),
        "limite": dados.get("limite"),
        "epocas": dados.get("epocas"),
    }
    model = CustomModel(canais, n_classes)
    model.load_state_dict(state_dict)
    return model.to(device).eval(), meta


def preprocessar_imagem(caminho, meta):
    # Mesma sequência do load_data: leitura, conversão de cor, resize, formato (C, H, W) e divisão pela escala
    flag = cv2.IMREAD_COLOR if meta["modo"] == "cor" else cv2.IMREAD_GRAYSCALE
    imagem = cv2.imread(str(caminho), flag)
    if imagem is None:
        raise ValueError(f"Não foi possível abrir a imagem: {caminho}")
    if meta["conversao_cor"] == "BGR2RGB":
        imagem = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB)
    elif meta["conversao_cor"] is not None:
        raise ValueError(f"Conversão de cor desconhecida: {meta['conversao_cor']}")
    imagem = cv2.resize(imagem, (meta["size"], meta["size"]))
    imagem = imagem.transpose(2, 0, 1) if meta["modo"] == "cor" else imagem[np.newaxis]
    return torch.from_numpy(imagem).float().unsqueeze(0) / meta["escala"]


def prever(caminho_imagem, model, meta, limiar):
    # A confiança do softmax não é calibrada: o limiar reduz erros, mas não garante acerto.
    # Em imagens diferentes das do treino, a rede pode errar com confiança alta e passar pelo limiar.
    entrada = preprocessar_imagem(caminho_imagem, meta).to(device)
    with torch.no_grad():
        probabilidades = torch.softmax(model(entrada), dim=1)[0].cpu().numpy()
    indice = int(probabilidades.argmax())
    confianca = float(probabilidades[indice])
    return {
        "classe": meta["class_names"][indice],
        "confianca": confianca,
        "probabilidades": dict(zip(meta["class_names"], probabilidades.tolist())),
        "status": "ok" if confianca >= limiar else "inconclusivo",
    }


def imagens_do_dataset(meta, inicio, quantidade):
    """Devolve (arquivo, índice da classe), por classe, a partir da posição inicio em ordem alfabética, como no load_data."""
    imagens = []
    if meta["cancer"] == "Pele":
        pasta = data_dir / "Skin_Cancer"
        tabela = pd.read_csv(pasta / "HAM10000_metadata.csv").sort_values("image_id")
        caminhos = {p.stem: p for parte in ("HAM10000_images_part_1", "HAM10000_images_part_2") for p in (pasta / parte).iterdir()}
        for indice, classe in enumerate(meta["class_names"]):
            ids = tabela.loc[tabela["dx"] == classe, "image_id"].tolist()[inicio:inicio + quantidade]
            imagens += [(caminhos[i], indice) for i in ids]
        return imagens

    pasta, prefixo = PASTAS_DATASET[meta["cancer"]]
    for indice, classe in enumerate(meta["class_names"]):
        arquivos = sorted(p for p in (data_dir / pasta / f"{prefixo}_{classe}").iterdir() if p.is_file())
        imagens += [(p, indice) for p in arquivos[inicio:inicio + quantidade]]
    return imagens


model, meta = carregar_modelo(modelos_dir / arquivo_modelo)
print(f"{arquivo_modelo}: {meta['cancer']} | {meta['modo']} | classes {meta['class_names']} | {meta['size']}x{meta['size']} | "
      f"conversão {meta['conversao_cor']} | escala {meta['escala']} | limite {meta['limite']} | épocas {meta['epocas']} | device {device}")

In [ ]:
# Teste de consistência: prever() precisa dar a mesma saída que o modelo recebendo os dados como no treino.
# Os modelos salvos são os finais, sem fold; por isso a comparação é com o caminho do load_data, não com a avaliação de um fold.
def lote_como_no_treino(caminhos, meta):
    # Mesmos passos do load_data e da célula do dataset dos notebooks
    flag = cv2.IMREAD_COLOR if meta["modo"] == "cor" else cv2.IMREAD_GRAYSCALE
    data_array = []
    for caminho in caminhos:
        image = cv2.imread(str(caminho), flag)
        if meta["conversao_cor"] == "BGR2RGB":
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        data_array.append(cv2.resize(image, (meta["size"], meta["size"])))
    features = np.asarray(data_array, dtype=np.uint8)
    features = features.transpose(0, 3, 1, 2) if meta["modo"] == "cor" else np.expand_dims(features, axis=1)
    return torch.from_numpy(features).float() / meta["escala"]


amostra = imagens_do_dataset(meta, 0, 5)
with torch.no_grad():
    saida_treino = torch.softmax(model(lote_como_no_treino([a for a, _ in amostra], meta).to(device)), dim=1).cpu().numpy()

maior_diferenca = 0.0
for (arquivo, real), esperado in zip(amostra, saida_treino):
    resultado = prever(arquivo, model, meta, limiar)
    obtido = np.array([resultado["probabilidades"][c] for c in meta["class_names"]])
    maior_diferenca = max(maior_diferenca, float(np.abs(obtido - esperado).max()))
    print(f"{arquivo.name}: real {meta['class_names'][real]} | prevista {resultado['classe']} | confiança {resultado['confianca']:.4f}")

if maior_diferenca > 1e-5:
    raise RuntimeError(f"Inconsistente: diferença máxima {maior_diferenca:.2e}. Confira o pré-processamento e os metadados.")
print(f"\nConsistente: {len(amostra)} imagens, diferença máxima {maior_diferenca:.2e}")

In [ ]:
# Classifica uma pasta de imagens novas
if not pasta_imagens.is_dir():
    raise FileNotFoundError(f"Crie a pasta {pasta_imagens.resolve()} e coloque as imagens nela")
arquivos = sorted(p for p in pasta_imagens.iterdir() if p.suffix.lower() in EXTENSOES)
if not arquivos:
    raise FileNotFoundError(f"Sem imagens em {pasta_imagens.resolve()}")

linhas = []
for arquivo in arquivos:
    try:
        resultado = prever(arquivo, model, meta, limiar)
        status = "ok" if resultado["status"] == "ok" else MENSAGEM_INCONCLUSIVO
        linhas.append({"arquivo": arquivo.name, "classe": resultado["classe"], "confiança_%": round(resultado["confianca"] * 100, 1), "status": status})
    except ValueError as erro:
        linhas.append({"arquivo": arquivo.name, "classe": "-", "confiança_%": None, "status": f"erro: {erro}"})

print(f"Modelo: {arquivo_modelo} | limiar: {limiar}\n")
print(pd.DataFrame(linhas).to_string(index=False))

In [ ]:
# Acurácia x cobertura: quanto o limiar melhora o acerto e quantas imagens deixam de ter resposta
def avaliar_conjunto(imagens):
    confiancas, acertos = [], []
    for arquivo, real in imagens:
        resultado = prever(arquivo, model, meta, limiar)
        confiancas.append(resultado["confianca"])
        acertos.append(resultado["classe"] == meta["class_names"][real])
    return np.array(confiancas), np.array(acertos)


conjuntos = {}

# Imagens depois do limite do treino: nunca foram usadas, mas vêm da mesma fonte, então a curva é otimista
inicio = meta["limite"] or 1000  # modelos antigos não guardam o limite; todos usaram 1000 por classe
reserva = imagens_do_dataset(meta, inicio, reserva_por_classe) if meta["cancer"] != "Pele" else []
if reserva:
    conjuntos["Mesma fonte, fora do treino"] = reserva
else:
    print("Sem imagens reservadas para este modelo: use o teste externo.")

externo = []
for indice, classe in enumerate(meta["class_names"]):
    pasta = teste_externo_dir / meta["id_cancer"] / classe
    if pasta.is_dir():
        externo += [(p, indice) for p in sorted(pasta.iterdir()) if p.suffix.lower() in EXTENSOES]
if externo:
    conjuntos["Outra fonte (teste externo)"] = externo

fig, ax = plt.subplots(figsize=(7, 4.5))
for nome, imagens in conjuntos.items():
    confiancas, acertos = avaliar_conjunto(imagens)
    linhas = []
    for valor in np.arange(0.50, 0.96, 0.05):
        respondidas = confiancas >= valor
        linhas.append({
            "limiar": round(float(valor), 2),
            "cobertura_%": round(float(respondidas.mean()) * 100, 1),
            "acc_nas_respondidas_%": round(float(acertos[respondidas].mean()) * 100, 1) if respondidas.any() else None,
            "inconclusivas": int((~respondidas).sum()),
            "erros_que_passaram": int((respondidas & ~acertos).sum()),
        })
    curva = pd.DataFrame(linhas)
    print(f"\n{nome} ({len(imagens)} imagens) | acerto sem limiar: {acertos.mean() * 100:.1f}%")
    print(curva.to_string(index=False))
    ax.plot(curva["cobertura_%"], curva["acc_nas_respondidas_%"], marker="o", label=nome)

ax.set_xlabel("Cobertura: imagens com resposta (%)")
ax.set_ylabel("Acurácia nas respondidas (%)")
ax.set_title(f"{arquivo_modelo}: acurácia x cobertura (limiares de 0,50 a 0,95)")
ax.grid()
if conjuntos:
    ax.legend()
plt.show()